In [0]:
from pyspark.sql.functions import lit, current_date

silver_customer = (
    spark.read.parquet("/Volumes/workspace/default/shree_project/silver/customers")
    .dropDuplicates(["customer_id"])
)

customer_dim = (
    silver_customer
    .withColumn("effective_from", current_date())
    .withColumn("effective_to", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
)

customer_dim.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/shree_project/gold/customer_dim_delta")

from delta.tables import DeltaTable

customer_delta = DeltaTable.forPath(
    spark,
    "/Volumes/workspace/default/shree_project/gold/customer_dim_delta"
)


incoming = (
    spark.read.parquet("/Volumes/workspace/default/shree_project/silver/customers")
    .dropDuplicates(["customer_id"])
)


### SCD merge

In [0]:
customer_delta.alias("t") \
.merge(
    incoming.alias("s"),
    "t.customer_id = s.customer_id AND t.is_current = true"
) \
.whenMatchedUpdate(
    condition="""
        t.name <> s.name OR
        t.email <> s.email OR
        t.phone <> s.phone OR
        t.city <> s.city OR
        t.state <> s.state
    """,
    set={
        "is_current": "false",
        "effective_to": "current_date()"
    }
) \
.whenNotMatchedInsert(
    values={
        "customer_id": "s.customer_id",
        "name": "s.name",
        "email": "s.email",
        "phone": "s.phone",
        "city": "s.city",
        "state": "s.state",
        "signup_date": "s.signup_date",
        "effective_from": "current_date()",
        "effective_to": "null",
        "is_current": "true"
    }
) \
.execute()
